In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks
# Obtener la ruta del directorio raíz del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))
# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(PROJECT_ROOT)

INPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'inputs')
OUTPUTS_FOLDER = os.path.join(PROJECT_ROOT, 'tests', 'test_data', 'outputs')
DOCUMENTS_FOLDER = os.path.join(PROJECT_ROOT, 'documents', 'inputs', 'test_files')

def get_file_from_path(bank:str, file_path: str) -> str:
    return os.path.join(DOCUMENTS_FOLDER, bank, file_path)

file_path = get_file_from_path('Banorte', 'Banorte_credit_new_statement.pdf')

In [2]:
from models import DocumentProcessingFacade
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

doc_processor = DocumentProcessingFacade(file_path)
statement_properties = doc_processor.get_statement_properties() 
bank = statement_properties['bank']
statement_type = statement_properties['statement_type']
new_format = statement_properties['new_format']

print(bank, ' - ', statement_type, ' - ', new_format)

BankType.BANORTE  -  StatementType.CREDIT  -  True


In [3]:
#extracted_words = doc_processor.get_extracted_words()
#
#if statement_type == 'debit':
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_extracted_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    extracted_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_extracted_words.csv')
#    
#extracted_words.to_csv(extracted_words_path, index=False)

In [4]:
corrected_extracted_words = doc_processor.get_corrected_extracted_words()
corrected_extracted_words

,page,text,x0,top,x1,bottom
0,2,2,553.038000,52.095053,557.486139,60.095303
1,2,de,559.703069,52.095053,568.599347,60.095303
2,2,8,570.831000,52.095053,575.279139,60.095303
3,2,MARIA,45.021750,72.287250,73.497750,81.287250
4,2,FERNANDA,75.990750,72.287250,125.454750,81.287250
...,...,...,...,...,...,...
4218,8,opinión,131.473888,754.353053,171.118789,765.353303
4219,8,de,174.165858,754.353053,186.992150,765.353303
4220,8,este,190.039219,754.353053,212.050719,765.353303
4221,8,comparativo:,215.097788,754.353053,284.124357,765.353303


In [5]:
#if statement_type == 'debit':
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_corrected_words.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    corrected_words_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_corrected_words.csv')
#    
#corrected_extracted_words.to_csv(corrected_words_path, index=False)

In [6]:
from models import TableProcessingFacade

table_processor = TableProcessingFacade(corrected_extracted_words, statement_properties)
reconstructor = table_processor.get_reconstructor()
structured_table = reconstructor.get_structured_table()
structured_table

,date,description,Monto
0,None,(NO A MESES),None
1,None,Tarjeta titular XXXX XXXX XXXX 4540,None
2,None,Fecha de Fecha de,None
3,None,Monto Descripción del movimiento,None
4,None,la operación cargo,None
5,10-SEP-2024,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,+$426.24
6,12-SEP-2024,AMAZON MX A MESES M CIUDAD DE M 02/03,"+$1,390.85"
7,13-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$56.00
8,17-SEP-2024,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,+$169.00
9,21-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$84.00


In [7]:
reconstructed_table = table_processor.reconstruct_table()
reconstructed_table

,date,description,Monto
0,10-SEP-2024,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,+$426.24
1,12-SEP-2024,AMAZON MX A MESES M CIUDAD DE M 02/03,"+$1,390.85"
2,13-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$56.00
3,17-SEP-2024,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,+$169.00
4,21-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$84.00
5,21-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$6.00
6,21-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$23.25
7,21-SEP-2024,"PAGO BANCA DIGITAL / SUCURSAL, GRACIAS.","-$5,630.57"
8,24-SEP-2024,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,+$95.00
9,25-SEP-2024,APPLE.COM/BILL 866-712-77 09/25/24 2.5017 USD...,+$49.25


In [8]:
#if statement_type == 'debit':
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_reconstructed_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    reconstructed_table_path = os.path.join(INPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_reconstructed_table.csv')
#    
#reconstructed_table.to_csv(reconstructed_table_path, index=False)

In [9]:
from models import DataProcessingFacade

data_processor = DataProcessingFacade(corrected_extracted_words, reconstructed_table, statement_properties)
period = data_processor.get_period()
print(period)

[datetime.date(2024, 9, 5), datetime.date(2024, 10, 4)]


In [10]:
df_transactions = data_processor.get_normalized_table()
df_transactions

,date,description,amount,type
0,2024-09-10,D LOCAL*DIDIFOOD CIUDAD DE MEX MX DMM 171208LT4,-426.24,Cargo
19,2024-09-11,SHEIN.COM SINGAPORE 09/11/24 85.8515 USD RT 1...,-1711.72,Cargo
1,2024-09-12,AMAZON MX A MESES M CIUDAD DE M 02/03,-1390.85,Cargo
2,2024-09-13,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-56.00,Cargo
3,2024-09-17,SIMILARES F3 OTE MERIDA YUC MX CCA 010830V77,-169.00,Cargo
4,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-84.00,Cargo
5,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-6.00,Cargo
6,2024-09-21,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-23.25,Cargo
7,2024-09-21,"PAGO BANCA DIGITAL / SUCURSAL, GRACIAS.",5630.57,Abono
8,2024-09-24,DLOCAL*DIDI RIDES CUAUHTEMOC MX DMM 171208LT4,-95.00,Cargo


In [11]:
#from models import CsvExporter
#
#exporter = CsvExporter(df_transactions)
#if statement_type == 'debit':
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_normalized_table.csv')
#elif statement_type == 'credit':
#    _format = 'new' if new_format else 'old'
#    normalized_table_path = os.path.join(OUTPUTS_FOLDER, f'{bank}_{statement_type}_{_format}_normalized_table.csv')
#
#exporter.export_data(normalized_table_path)